# Notebook 2: Diffusion Models & Flow Matching

In this notebook you will implement two generative paradigms from their mathematical foundations:

1. **DDPM Forward Process** — adding noise to data
2. **DDPM Reverse Denoising Loop** — iteratively removing noise
3. **Flow Matching Interpolation** — the straight-path formulation
4. **Flow Matching Training Step** — velocity prediction with MSE loss
5. **Euler Integration Sampling** — generating samples from a trained flow model

Each exercise has `# TODO` blocks to fill in and `assert`-based tests to verify correctness.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
DEVICE = "cpu"

---
## Exercise 2a: DDPM Forward Process

In Denoising Diffusion Probabilistic Models (DDPM), the forward process adds Gaussian noise to data over $T$ timesteps.

**Linear beta schedule:**
$$\beta_t = \beta_{\min} + \frac{t}{T-1}(\beta_{\max} - \beta_{\min}), \quad t = 0, \dots, T-1$$

**Cumulative products:**
$$\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=0}^{t} \alpha_s$$

**Forward sampling (closed form):**
$$q(x_t | x_0) = \mathcal{N}\!\left(x_t; \sqrt{\bar{\alpha}_t}\, x_0,\; (1-\bar{\alpha}_t)\, I\right)$$
$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

In [ ]:
def linear_beta_schedule(T: int, beta_min: float = 1e-4, beta_max: float = 0.02) -> torch.Tensor:
    """
    Compute a linear beta schedule.

    Returns
    -------
    betas : Tensor of shape [T]
    """
    # TODO: Return a linearly spaced tensor from beta_min to beta_max with T steps
    raise NotImplementedError("Implement linear_beta_schedule")


def compute_alpha_bar(betas: torch.Tensor) -> torch.Tensor:
    """
    Compute alpha_bar_t = cumulative product of (1 - beta_s) for s=0..t.

    Returns
    -------
    alpha_bar : Tensor of shape [T]
    """
    # TODO:
    # 1. alphas = 1 - betas
    # 2. alpha_bar = cumulative product of alphas
    raise NotImplementedError("Implement compute_alpha_bar")


def ddpm_forward(
    x_0: torch.Tensor,       # [B, D] clean data
    t: torch.Tensor,          # [B] integer timesteps in [0, T-1]
    alpha_bar: torch.Tensor,  # [T] precomputed alpha_bar schedule
    noise: torch.Tensor = None,
) -> tuple:
    """
    Sample x_t from q(x_t | x_0).

    Returns
    -------
    x_t : Tensor [B, D]
    noise : Tensor [B, D]  (the epsilon that was used)
    """
    # TODO:
    # 1. If noise is None, sample noise ~ N(0, I) same shape as x_0
    # 2. Gather alpha_bar_t for each sample: alpha_bar_t = alpha_bar[t]  -> [B]
    # 3. Reshape alpha_bar_t for broadcasting: [B, 1] (or [B, 1, ...] matching x_0)
    # 4. x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
    # 5. Return (x_t, noise)
    raise NotImplementedError("Implement ddpm_forward")

In [ ]:
# === Tests for Exercise 2a ===
T = 1000
betas = linear_beta_schedule(T)
alpha_bar = compute_alpha_bar(betas)

# Schedule properties
assert betas.shape == (T,), f"betas shape: {betas.shape}"
assert alpha_bar.shape == (T,), f"alpha_bar shape: {alpha_bar.shape}"
assert torch.allclose(betas[0], torch.tensor(1e-4), atol=1e-6), "betas[0] should be beta_min"
assert torch.allclose(betas[-1], torch.tensor(0.02), atol=1e-4), "betas[-1] should be beta_max"
assert alpha_bar[0] > alpha_bar[-1], "alpha_bar should be decreasing"
assert alpha_bar[0] < 1.0 and alpha_bar[0] > 0.99, "alpha_bar[0] should be close to 1"
assert alpha_bar[-1] < 0.1, "alpha_bar[T-1] should be close to 0"

# Forward process
torch.manual_seed(0)
x_0 = torch.randn(4, 2)

# At t=0, x_t should be very close to x_0
x_t_0, eps_0 = ddpm_forward(x_0, torch.zeros(4, dtype=torch.long), alpha_bar)
assert torch.allclose(x_t_0, x_0, atol=0.02), "At t=0, x_t should be ≈ x_0"

# At t=T-1, x_t should be nearly pure noise
x_t_T, eps_T = ddpm_forward(x_0, torch.full((4,), T - 1, dtype=torch.long), alpha_bar)
assert (x_t_T - x_0).abs().mean() > 0.5, "At t=T-1, x_t should differ significantly from x_0"

# Shape check
assert x_t_0.shape == x_0.shape
assert eps_0.shape == x_0.shape

print("Exercise 2a passed!")

---
## Exercise 2b: DDPM Reverse Denoising Loop

Given a trained noise-prediction network $\epsilon_\theta(x_t, t)$, the reverse process generates data from noise.

**Predicted $x_0$:**
$$\hat{x}_0 = \frac{x_t - \sqrt{1 - \bar{\alpha}_t}\, \epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}}$$

**Posterior mean:**
$$\mu_\theta(x_t, t) = \frac{\sqrt{\bar{\alpha}_{t-1}}\, \beta_t}{1 - \bar{\alpha}_t} \hat{x}_0 + \frac{\sqrt{\alpha_t}(1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t} x_t$$

**Posterior variance:**
$$\sigma_t^2 = \frac{(1 - \bar{\alpha}_{t-1})}{(1 - \bar{\alpha}_t)} \beta_t$$

**Sample:**
$$x_{t-1} = \mu_\theta(x_t, t) + \sigma_t \, z, \quad z \sim \mathcal{N}(0, I) \text{ (except at } t=0\text{)}$$

In [ ]:
# Helper: a tiny noise-prediction network for testing
class TinyEpsNet(nn.Module):
    """Small MLP that predicts noise given (x_t, t_embedding)."""
    def __init__(self, data_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(16, hidden_dim),
            nn.GELU(),
        )
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, x_t, t_emb):
        t_feat = self.time_mlp(t_emb)
        return self.net(torch.cat([x_t, t_feat], dim=-1))


def sinusoidal_embedding(t: torch.Tensor, dim: int = 16) -> torch.Tensor:
    """Simple sinusoidal embedding for integer timesteps."""
    half = dim // 2
    freqs = torch.exp(-math.log(1000.0) * torch.arange(half, device=t.device).float() / half)
    args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

In [ ]:
def ddpm_denoise_step(
    eps_net: nn.Module,
    x_t: torch.Tensor,          # [B, D]
    t_idx: int,                  # current integer timestep
    betas: torch.Tensor,         # [T]
    alpha_bar: torch.Tensor,     # [T]
) -> torch.Tensor:
    """
    One reverse step: x_t -> x_{t-1}.

    Returns
    -------
    x_{t-1} : Tensor [B, D]
    """
    # TODO:
    # 1. Compute t_emb = sinusoidal_embedding of t_idx (broadcast to batch)
    # 2. Predict noise: eps_pred = eps_net(x_t, t_emb)
    # 3. Get alpha_t = 1 - betas[t_idx], alpha_bar_t = alpha_bar[t_idx]
    # 4. Get alpha_bar_prev = alpha_bar[t_idx - 1] if t_idx > 0 else 1.0
    # 5. Compute predicted x_0: x0_pred = (x_t - sqrt(1 - alpha_bar_t) * eps_pred) / sqrt(alpha_bar_t)
    # 6. Compute posterior mean:
    #    coeff1 = sqrt(alpha_bar_prev) * betas[t_idx] / (1 - alpha_bar_t)
    #    coeff2 = sqrt(alpha_t) * (1 - alpha_bar_prev) / (1 - alpha_bar_t)
    #    mu = coeff1 * x0_pred + coeff2 * x_t
    # 7. Compute posterior variance:
    #    sigma_sq = (1 - alpha_bar_prev) / (1 - alpha_bar_t) * betas[t_idx]
    # 8. If t_idx > 0: x_{t-1} = mu + sqrt(sigma_sq) * z, z ~ N(0,I)
    #    If t_idx == 0: x_{t-1} = mu  (no noise at final step)
    raise NotImplementedError("Implement ddpm_denoise_step")


@torch.no_grad()
def ddpm_sample(
    eps_net: nn.Module,
    shape: tuple,                # (B, D)
    T: int,
    betas: torch.Tensor,
    alpha_bar: torch.Tensor,
) -> torch.Tensor:
    """
    Full reverse sampling loop: start from x_T ~ N(0,I), iterate to x_0.

    Returns
    -------
    x_0 : Tensor [B, D]
    """
    # TODO:
    # 1. Sample x_T ~ N(0, I) with the given shape
    # 2. For t = T-1, T-2, ..., 0:
    #       x = ddpm_denoise_step(eps_net, x, t, betas, alpha_bar)
    # 3. Return final x
    raise NotImplementedError("Implement ddpm_sample")

In [ ]:
# === Tests for Exercise 2b ===
torch.manual_seed(10)
T_test = 50
betas_test = linear_beta_schedule(T_test)
abar_test = compute_alpha_bar(betas_test)

data_dim = 2
eps_net = TinyEpsNet(data_dim).eval()

# Single step shouldn't crash and should preserve shape
x_t = torch.randn(8, data_dim)
x_prev = ddpm_denoise_step(eps_net, x_t, t_idx=10, betas=betas_test, alpha_bar=abar_test)
assert x_prev.shape == (8, data_dim), f"Expected (8, {data_dim}), got {x_prev.shape}"

# Full sampling loop
samples = ddpm_sample(eps_net, shape=(16, data_dim), T=T_test, betas=betas_test, alpha_bar=abar_test)
assert samples.shape == (16, data_dim), f"Expected (16, {data_dim}), got {samples.shape}"
assert torch.isfinite(samples).all(), "Samples should be finite"

print("Exercise 2b passed!")

---
## Exercise 2c: Flow Matching — Forward Interpolation

Flow matching defines a straight-line path from data $x_0$ (at $t=0$) to noise $\epsilon$ (at $t=1$):

$$x_t = t \cdot \epsilon + (1 - t) \cdot x_0$$

The target velocity field along this path is constant:

$$u_t = \frac{dx_t}{dt} = \epsilon - x_0$$

This is the formulation used in the SimVLA codebase for robot action prediction.

In [ ]:
def flow_matching_interpolate(
    x_0: torch.Tensor,     # [B, D] clean data
    noise: torch.Tensor,   # [B, D] ~ N(0, I)
    t: torch.Tensor,       # [B] in (0, 1)
) -> tuple:
    """
    Compute the interpolated state x_t and the target velocity u_t.

    Returns
    -------
    x_t : Tensor [B, D]
    u_t : Tensor [B, D]  — target velocity
    """
    # TODO:
    # 1. Reshape t to [B, 1] for broadcasting
    # 2. x_t = t * noise + (1 - t) * x_0
    # 3. u_t = noise - x_0
    # 4. Return (x_t, u_t)
    raise NotImplementedError("Implement flow_matching_interpolate")

In [ ]:
# === Tests for Exercise 2c ===
torch.manual_seed(5)
B, D = 8, 2
x_0 = torch.randn(B, D)
eps = torch.randn(B, D)

# At t=0, x_t should equal x_0
t_zero = torch.zeros(B)
x_t, u_t = flow_matching_interpolate(x_0, eps, t_zero)
assert torch.allclose(x_t, x_0, atol=1e-6), "At t=0, x_t = x_0"

# At t=1, x_t should equal noise
t_one = torch.ones(B)
x_t, u_t = flow_matching_interpolate(x_0, eps, t_one)
assert torch.allclose(x_t, eps, atol=1e-6), "At t=1, x_t = noise"

# Velocity should be constant (noise - x_0) regardless of t
t_half = torch.full((B,), 0.5)
_, u_half = flow_matching_interpolate(x_0, eps, t_half)
_, u_quarter = flow_matching_interpolate(x_0, eps, torch.full((B,), 0.25))
assert torch.allclose(u_half, u_quarter, atol=1e-6), "Velocity u_t should be independent of t"
assert torch.allclose(u_half, eps - x_0, atol=1e-6), "u_t should equal noise - x_0"

# Shape
assert x_t.shape == (B, D) and u_t.shape == (B, D)

print("Exercise 2c passed!")

---
## Exercise 2d: Flow Matching Training Step

A single training step for conditional flow matching:

1. Sample mini-batch $x_0$ from data
2. Sample $t \sim \text{Beta}(1.5, 1) \cdot 0.999 + 0.001$ (avoiding exact 0 and 1)
3. Sample $\epsilon \sim \mathcal{N}(0, I)$
4. Compute $x_t$ and target velocity $u_t$
5. Predict velocity: $v_\theta(x_t, t)$
6. Loss: $\mathcal{L} = \| v_\theta - u_t \|^2$

The Beta(1.5, 1) distribution biases sampling toward $t \approx 1$ (noisier states), which helps training.

In [ ]:
# Helper: a tiny velocity network for testing
class TinyVelocityNet(nn.Module):
    """Small MLP that predicts velocity given (x_t, t)."""
    def __init__(self, data_dim: int, time_dim: int = 16, hidden_dim: int = 128):
        super().__init__()
        self.time_dim = time_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden_dim),
            nn.GELU(),
        )
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        half = self.time_dim // 2
        freqs = torch.exp(-math.log(100.0) * torch.arange(half, device=t.device).float() / half)
        args = t[:, None] * freqs[None]
        t_emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        t_feat = self.time_mlp(t_emb)
        return self.net(torch.cat([x_t, t_feat], dim=-1))

In [ ]:
def flow_matching_train_step(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    x_0: torch.Tensor,   # [B, D] — batch of clean data
) -> float:
    """
    One flow matching training step.

    Returns
    -------
    loss_value : float
    """
    # TODO:
    # 1. Sample t ~ Beta(1.5, 1.0) for each sample in batch, shape [B]
    #    Scale: t = t * 0.999 + 0.001
    # 2. Sample noise ~ N(0, I), same shape as x_0
    # 3. Compute x_t and u_t using flow_matching_interpolate
    # 4. Predict velocity: v_pred = model(x_t, t)
    # 5. Compute MSE loss: loss = mean((v_pred - u_t)^2)
    # 6. Backprop: optimizer.zero_grad(), loss.backward(), optimizer.step()
    # 7. Return loss.item()
    raise NotImplementedError("Implement flow_matching_train_step")

In [ ]:
# === Tests for Exercise 2d ===
torch.manual_seed(42)
data_dim = 2
model = TinyVelocityNet(data_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Generate a simple 2D dataset: circle
theta = torch.linspace(0, 2 * math.pi, 256)
data = torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)  # [256, 2]

# Run a few training steps and check loss decreases
losses = []
for i in range(200):
    idx = torch.randint(0, len(data), (64,))
    batch = data[idx]
    loss = flow_matching_train_step(model, optimizer, batch)
    losses.append(loss)

assert losses[0] > losses[-1], f"Loss should decrease: {losses[0]:.4f} -> {losses[-1]:.4f}"
assert losses[-1] < losses[0] * 0.8, "Loss should decrease by at least 20%"

print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
print("Exercise 2d passed!")

---
## Exercise 2e: Euler Integration Sampling

To generate samples from a trained flow model, integrate the learned velocity field from $t=1$ (noise) to $t=0$ (data) using Euler's method:

$$x_{t+\Delta t} = x_t + \Delta t \cdot v_\theta(x_t, t)$$

Starting at $x_1 \sim \mathcal{N}(0, I)$ and stepping with $\Delta t = -1/N$ for $N$ steps.

This is the exact procedure used in `SmolVLMVLA.generate_actions` for robot action inference.

In [ ]:
@torch.no_grad()
def euler_sample(
    model: nn.Module,
    shape: tuple,         # (B, D)
    num_steps: int = 10,  # number of Euler steps
) -> torch.Tensor:
    """
    Generate samples via Euler integration of the learned velocity field.

    Integrate from t=1 (noise) to t=0 (data).

    Returns
    -------
    x_0 : Tensor of shape `shape`
    """
    # TODO:
    # 1. x = randn(shape)  — start from pure noise
    # 2. dt = -1.0 / num_steps
    # 3. t = 1.0  (start at noise)
    # 4. For step in range(num_steps):
    #      a. Create t_batch = tensor of shape [B] filled with current t
    #      b. v = model(x, t_batch)  — predict velocity
    #      c. x = x + dt * v         — Euler step
    #      d. t = t + dt             — advance time
    # 5. Return x
    raise NotImplementedError("Implement euler_sample")

In [ ]:
# === Tests for Exercise 2e ===
torch.manual_seed(99)
model.eval()

samples = euler_sample(model, shape=(200, 2), num_steps=20)

# Shape check
assert samples.shape == (200, 2), f"Expected (200, 2), got {samples.shape}"

# Samples should be finite
assert torch.isfinite(samples).all(), "Samples should be finite"

# Samples should be roughly in the range of the training data (circle of radius 1)
norms = samples.norm(dim=-1)
assert norms.mean() < 3.0, f"Mean norm {norms.mean():.2f} too large — samples may not have converged"

# More steps should give different (hopefully better) results than fewer steps
samples_5 = euler_sample(model, shape=(50, 2), num_steps=5)
samples_50 = euler_sample(model, shape=(50, 2), num_steps=50)
assert samples_5.shape == samples_50.shape

print("Exercise 2e passed!")
print("\n=== All Notebook 2 exercises passed! ===")

In [ ]:
# Optional visualization: plot generated samples vs training data
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(data[:, 0], data[:, 1], alpha=0.3, s=5, label="Training data")
axes[0].set_title("Training Data (circle)")
axes[0].set_xlim(-2, 2); axes[0].set_ylim(-2, 2)
axes[0].set_aspect("equal")

s = euler_sample(model, shape=(500, 2), num_steps=50).numpy()
axes[1].scatter(s[:, 0], s[:, 1], alpha=0.3, s=5, c="orange", label="Generated")
axes[1].set_title("Flow Matching Samples")
axes[1].set_xlim(-2, 2); axes[1].set_ylim(-2, 2)
axes[1].set_aspect("equal")

plt.tight_layout()
plt.show()